**Amazon Sales Dataset Analysis**

**Background:**

    I chose a less complex dataset because I was learning Power BI at the time and wanted to practice with manageable data. It then occurred to me that I could also use this dataset to perform some descriptive and prescriptive analysis. Hence, this project was created. The purpose is to showcase structured business thinking and demonstrate technical skills in data manipulation.

**Data:**

    The data was sourced from Kaggle and contains 250 rows and 11 columns. It is a simple dataset based on fictional Amazon platform orders, including information such as product, price, sales, order status, and other relevant details.

**Business Questions:**

    Through e-commerce transactions, we can evaluate profitability and consumer behavior, which are key drivers of e-commerce performance. Therefore, the business questions we aim to answer are:

    1. Which product categories and individual products drive the most revenue on the platform?
    2. What is the order completion rate (percentage of completed orders vs. total orders placed) on this platform?

**Source:** 
    https://www.kaggle.com/datasets/pankajpatil7721/amazon-sales-data-2025

--------------------------------------------------------------------------------------------------------------------------------

Let's begin by importing the necessary libraries for this project.

In [4]:
import pandas as pd
import numpy as np

Next, we load the dataset into the notebook

In [6]:
data = pd.read_csv(r"C:\Users\JT-Cargo\Desktop\Courses\Projects\Amazon Sales\amazon_sales_data 2025.csv", encoding = 'utf-8', header = 0)

We start with a data quality check to ensure there are no duplicates or null values. This step helps prevent abnormalities during the analysis process.

In [8]:
#Data Quality Check

data.isnull().sum()
data.duplicated().sum()

0

Next, we proceed with data cleaning. Since there are no duplicates or null values, we pay attention to the data types. It turns out the Date columns are not converted to datetime yet. To maintain data quality, we convert them to datetime.

In [10]:
#Data Cleaning

data['Date'] = pd.to_datetime(data['Date'], format = '%d-%m-%y')

Before we start the sales analysis, there are several additional pieces of information that are necessary to aid the analysis. Hence, we create new columns to extract this information based on the data provided.

Here is the thought process for why we extract the extra columns:

1. **Revenue (Earned, Deferred, and Cancelled Revenue/Quantity)** - In the raw dataset, revenue/quantity are not classified. Classification simplifies the calculation process, hence increasing processing speed.
2. **Expected Revenue** - Since there are completed and uncompleted revenue/quantity, this column is created to simply estimate the potential earned revenue/completed quantity.
3. **Month** - Enables time-based analysis.

In [12]:
#Additional Information

data['Earned Revenue'] = np.where(data['Status'] == 'Completed',data['Total Sales'],0)
data['Deferred Revenue'] = np.where(data['Status'] == 'Pending',data['Total Sales'],0)
data['Cancelled Revenue'] = np.where(data['Status'] == 'Cancelled', data['Total Sales'],0)
data['Expected Revenue'] = np.where(data['Status'].isin(['Completed','Pending']),data['Total Sales'],0)
data['Completed Quantity'] = np.where(data['Status'] == 'Completed',data['Quantity'],0)
data['Pending Quantity'] = np.where(data['Status'] == 'Pending',data['Quantity'],0)
data['Cancelled Quantity'] = np.where(data['Status'] == 'Cancelled', data['Quantity'],0)
data['Expected Quantity'] = np.where(data['Status'].isin(['Completed','Pending']),data['Quantity'],0)
data['Expected Order'] = np.where(data['Status'].isin(['Completed','Pending']),1,0)
data['Pending Order'] = np.where(data['Status'] == 'Pending',1,0)
data['Cancelled Order'] = np.where(data['Status'] == 'Cancelled',1,0)
data['Month'] = data['Date'].dt.month

After cleaning and extracting additional information, here is the final dataframe

In [14]:
#Data Frame
data

,Order ID,Date,Product,Category,Price,Quantity,Total Sales,Customer Name,Customer Location,Payment Method,...,Cancelled Revenue,Expected Revenue,Completed Quantity,Pending Quantity,Cancelled Quantity,Expected Quantity,Expected Order,Pending Order,Cancelled Order,Month
0,ORD0001,2025-03-14,Running Shoes,Footwear,60,3,180,Emma Clark,New York,Debit Card,...,180,0,0,0,3,0,0,0,1,3
1,ORD0002,2025-03-20,Headphones,Electronics,100,4,400,Emily Johnson,San Francisco,Debit Card,...,0,400,0,4,0,4,1,1,0,3
2,ORD0003,2025-02-15,Running Shoes,Footwear,60,2,120,John Doe,Denver,Amazon Pay,...,120,0,0,0,2,0,0,0,1,2
3,ORD0004,2025-02-19,Running Shoes,Footwear,60,3,180,Olivia Wilson,Dallas,Credit Card,...,0,180,0,3,0,3,1,1,0,2
4,ORD0005,2025-03-10,Smartwatch,Electronics,150,3,450,Emma Clark,New York,Debit Card,...,0,450,0,3,0,3,1,1,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,ORD0246,2025-03-17,T-Shirt,Clothing,20,2,40,Daniel Harris,Miami,Debit Card,...,40,0,0,0,2,0,0,0,1,3
246,ORD0247,2025-03-30,Jeans,Clothing,40,1,40,Sophia Miller,Dallas,Debit Card,...,40,0,0,0,1,0,0,0,1,3
247,ORD0248,2025-03-05,T-Shirt,Clothing,20,2,40,Chris White,Denver,Debit Card,...,40,0,0,0,2,0,0,0,1,3
248,ORD0249,2025-03-08,Smartwatch,Electronics,150,3,450,Emily Johnson,New York,Debit Card,...,450,0,0,0,3,0,0,0,1,3


We start the analysis by finding general metrics. This helps us grasp the overall sales situation.

**Note:** Despite adding new columns to ease the calculation, I still try to calculate using conditional calculations to demonstrate my technical understanding.

In [16]:
#General Metric

Earned_Revenue = data[data['Status'] == 'Completed']['Total Sales'].sum()
Deferred_Revenue = data['Deferred Revenue'].sum()
Expected_Revenue = data['Expected Revenue'].sum()
Cancelled_Revenue = data['Cancelled Revenue'].sum()
Product_Category = data['Category'].nunique()
Product_Sold = data['Product'].nunique()
Total_Customer = data['Customer Name'].nunique()
Total_Order = data['Order ID'].count()
Completed_Order = data[data['Status'] == 'Completed']['Status'].count()
Completion_Rate = (Completed_Order/Total_Order)*100
Cancelled_Order = data[data['Status'] == 'Cancelled']['Status'].count()
Cancellation_Rate = (Cancelled_Order/Total_Order)*100
Pending_Order = data[data['Status'] == 'Pending']['Status'].count()
Pending_Rate = (Pending_Order/Total_Order)*100
Average_Order_Value = (Earned_Revenue/Completed_Order).round(2)

General_Metric = pd.DataFrame(data = {
    'Earned Revenue' : [Earned_Revenue],
    'Deferred Revenue' : [Deferred_Revenue],
    'Expected Revenue' : [Expected_Revenue],
    'Cancelled Revenue' : [Cancelled_Revenue],
    'Total Category' : [Product_Category],
    'Product Sold' : [Product_Sold],
    'Total Customer': [Total_Customer],
    'Total Order': [Total_Order],
    'Average Order Value': [Average_Order_Value],
    'Completed Order' : [Completed_Order],
    'Completion Rate (%)' : [Completion_Rate],
    'Cancelled Order' : [Cancelled_Order],
    'Cancellation Rate (%)' : [Cancellation_Rate],
    'Pending Order' : [Pending_Order],
    'Pending Rate (%)' : [Pending_Rate]})

In [17]:
General_Metric

,Earned Revenue,Deferred Revenue,Expected Revenue,Cancelled Revenue,Total Category,Product Sold,Total Customer,Total Order,Average Order Value,Completed Order,Completion Rate (%),Cancelled Order,Cancellation Rate (%),Pending Order,Pending Rate (%)
0,88530,90285,178815,65030,5,10,10,250,1006.02,88,35.2,77,30.8,85,34.0


From the dataframe, we can infer several insights:

**1. Revenue Analysis**

The company earned **\\$88,530**; however, there are **\\$90,285** worth of transactions that need to be followed through. Considering deferred revenue consists of more than 50% of the expected revenue, it is recommended that the platform ensures these transactions are completed. This could be done through several means:

* **Customer Service** - Active communication and real-time tracking to ensure customers can track their orders
* **Delivery** - Ensure a smooth delivery process from the first mile, middle mile, and last mile. Prevent returns or cancellations due to damages or tardiness
* **Post-Delivery** - Points or rewards for customers who complete their transactions. Swift aftersales service is also necessary to maintain customer satisfaction

This is quite urgent, considering the **cancellation rate is high (30.8%)** , meaning the possibility of pending orders being cancelled is high. Failure to follow through on pending transactions will cause the platform to lose potential revenue.

**2. Cancellation Analysis**

The cancellation rate is staggering—almost one-third of total orders—causing the platform to **lose $65,030 (42.3%) of its earnings**. This indicates the existence of pain points that lead to customer cancellations. There are several areas we could investigate:

**Pre-Shipping**

a. **Platform User Interface or User Experience** - The platform's UI/UX must be convenient for both sellers and buyers.

* For sellers: Simplifying the inventory update process could prevent out-of-stock products from being displayed. Clear and concise order information could also help sellers through the packaging process.

* For buyers: Providing clear and concise information prevents confusion or purchases by mistake (wrong colors, quantity, type, etc.). Allowing buyers to track their items also helps build trust in the shipping process.

b. **Pick-up Arrangement** - The pick-up arrangement process must be swift to speed up the shipping process. This could be ensured by having sellers pack orders promptly and having them picked up by 3PL (Third Party Logistics) or in-house logistics within a certain time frame after the buyer places the order.

**Post-Shipping**

a. **Logistics SLA (Service Level Agreement)** - The SLA determines the time taken to deliver from one point to another. The platform's logistics service must comply with the agreed SLA.

b. **On-Loading and Off-Loading Process** - Most items are broken due to mishandling during the delivery process. Hence, Standard Operating Procedures for on-loading and off-loading processes matter.

If items are delivered swiftly without damage, it might increase customer satisfaction and prevent cancellations.

-------------------------------------------------------------------------------------------------------------------------------

Next, we will break the analysis down into more detail, starting with customers.

**1. Customer Analysis**

Here is the thought process for the dataframe created:

* **Customer Product Metric** - Created to identify customer purchasing behavior

* **Top Category** - Identifies each customer's most purchased category (by quantity)

* **Top Spending** - Identifies each customer's highest spending category (by revenue)

Understanding both quantity purchased and total spending is important. **Some categories might be purchased in large quantities but generate small revenue, while other items might bring in more revenue despite being purchased in smaller quantities**. Hence, I also created a column for price per item to understand which items generate more revenue and help adjust our strategy accordingly.

In [21]:
#Customer Metric

Customer_Product_Metric = data[data['Status'].isin(['Completed','Pending'])].groupby(['Customer Name', 'Category']).agg(
    Total_Items_Purchased = ('Quantity','sum'),
    Total_Expected_Revenue = ('Total Sales', 'sum'),
    Revenue_per_Item = ('Price','mean'),
    Pending_Items = ('Pending Quantity','sum'),
    Deferred_Revenue = ('Deferred Revenue','sum')).reset_index().round(2)

Top_Category = (
    Customer_Product_Metric
    .sort_values(['Customer Name','Total_Items_Purchased'], ascending = [True, False])
    .drop_duplicates('Customer Name',keep = 'first')
    .reset_index(drop = True)
    .rename(columns = {'Category' : 'Most Purchased Category'})
)

Top_Spending = (
    Customer_Product_Metric
    .sort_values(['Customer Name','Total_Expected_Revenue'], ascending = [True, False])
    .drop_duplicates('Customer Name',keep = 'first')
    .reset_index(drop = True)
    .rename(columns = {'Category' : 'Most Spent Category'})
)

In [22]:
Customer_Metric = data.groupby('Customer Name').agg(
    Expected_Revenue = ('Expected Revenue', 'sum'),
    Total_Earned_Revenue = ('Earned Revenue', 'sum'),
    Total_Deferred_Revenue = ('Deferred Revenue','sum'),
    Cancelled_Revenue = ('Cancelled Revenue', 'sum'),
    Item_Purchased = ('Quantity','sum'),
    Order = ('Order ID','size'),
    Completion_Rate = ('Status', lambda x : ((x == 'Completed').sum()/len(x)*100).round(2)),
    Cancellation_Rate = ('Status', lambda x : ((x == 'Cancelled').sum()/len(x)*100).round(2)),
    Pending_Rate = ('Status', lambda x : ((x == 'Pending').sum()/len(x)*100).round(2))).reset_index()

Customer = (
    Customer_Metric
    .merge(Top_Category[['Customer Name','Most Purchased Category','Total_Items_Purchased']], on = 'Customer Name',how = 'left')
    .merge(Top_Spending[['Customer Name','Most Spent Category','Total_Expected_Revenue','Revenue_per_Item','Pending_Items','Deferred_Revenue']], on = 'Customer Name', how = 'left')
)

Customer.sort_values('Total_Deferred_Revenue', ascending = False)

,Customer Name,Expected_Revenue,Total_Earned_Revenue,Total_Deferred_Revenue,Cancelled_Revenue,Item_Purchased,Order,Completion_Rate,Cancellation_Rate,Pending_Rate,Most Purchased Category,Total_Items_Purchased,Most Spent Category,Total_Expected_Revenue,Revenue_per_Item,Pending_Items,Deferred_Revenue
8,Olivia Wilson,30495,13460,17035,5675,83,29,37.93,27.59,34.48,Electronics,30,Electronics,15300,475.00,15,5900
1,Daniel Harris,14545,2670,11875,4400,66,23,26.09,30.43,43.48,Electronics,27,Electronics,9350,344.44,18,8100
0,Chris White,13060,2110,10950,5825,56,22,31.82,31.82,36.36,Electronics,18,Electronics,7650,385.71,15,6150
2,David Lee,14430,4550,9880,8235,72,26,11.54,42.31,46.15,Electronics,20,Home Appliances,8400,1200.00,4,4800
9,Sophia Miller,10315,2725,7590,2980,42,16,43.75,25.00,31.25,Electronics,18,Electronics,5000,316.67,11,2550
7,Michael Brown,17175,9905,7270,5480,75,24,41.67,33.33,25.00,Electronics,26,Electronics,11000,392.86,16,7050
5,Jane Smith,14345,7250,7095,16840,88,30,23.33,43.33,33.33,Clothing,20,Electronics,8950,458.33,8,3600
6,John Doe,22240,15285,6955,4630,71,26,46.15,19.23,34.62,Electronics,33,Electronics,11650,366.67,14,6700
3,Emily Johnson,16205,9855,6350,7270,66,22,40.91,36.36,22.73,Electronics,30,Electronics,10950,375.00,6,1350
4,Emma Clark,26005,20720,5285,3695,95,32,50.00,18.75,31.25,Electronics,36,Electronics,17750,481.82,4,1250


From the dataframe above, we can infer several key insights:

**1.** While Emma Clark contributes the highest revenue (\\$20,720), **Olivia Wilson is expected to contribute more if she follows through with all transactions (\\$30,495)**. This highlights the importance of accommodating customer needs as explained before, since it affects profit.

**2.** David Lee and Jane Smith have **high cancellation rates**. This serves as an opportunity to check their transaction history and identify the root cause of their cancellations. They could also be **ideal respondents** for a survey **to understand** how the platform can **improve and prevent future cancellations**.

**3.** We can identify each customer's most purchased and most spent categories. This allows us to **tailor personalized ads** for each customer or provide discounts on purchases in those specific categories. Considering **electronics is the leading category** for most customers, we could set up an event that offers **promotions or special prices on electronics products**.

-------------------------------------------------------------------------------------------------------------------------------

**2. Category and Product Analysis**

Here is the thought process for the dataframe created:

* **Category** - To understand platform performance by its catalog

* **Product** - To break down the category analysis into specific items and understand which items affect each category's performance

We need to understand how our current item catalog generates revenue for the platform. By identifying the relationship between each category and product with the platform's revenue, strategies can be adjusted accordingly depending on the products.

In [26]:
Category = data.groupby('Category').agg(
    Total_Expected_Revenue = ('Expected Revenue','sum'),
    Active_and_Completed_Order = ('Expected Order','sum'),
    Total_Earned_Revenue = ('Earned Revenue', 'sum'),
    Total_Deferred_Revenue = ('Deferred Revenue','sum'),
    Cancelled_Revenue = ('Cancelled Revenue', 'sum'),
    Expected_Item_Purchased = ('Expected Quantity','sum'),
    Order = ('Order ID','size'),
    Completion_Rate = ('Status', lambda x : ((x == 'Completed').sum()/len(x)*100).round(2)),
    Cancellation_Rate = ('Status', lambda x : ((x == 'Cancelled').sum()/len(x)*100).round(2)),
    Pending_Rate = ('Status', lambda x : ((x == 'Pending').sum()/len(x)*100).round(2))).reset_index()

In [27]:
Category['Average Order Value'] = (Category['Total_Expected_Revenue']/Category['Active_and_Completed_Order']).round(2)
(
    Category
    .sort_values('Total_Expected_Revenue', ascending = False)
    .reset_index(drop = True)
    [['Category','Total_Expected_Revenue','Active_and_Completed_Order','Average Order Value','Pending_Rate','Cancellation_Rate']]
)

,Category,Total_Expected_Revenue,Active_and_Completed_Order,Average Order Value,Pending_Rate,Cancellation_Rate
0,Electronics,103300,85,1215.29,34.75,27.97
1,Home Appliances,69000,24,2875.00,27.50,40.00
2,Footwear,3240,19,170.53,33.33,29.63
3,Clothing,2420,27,89.63,30.00,32.50
4,Books,855,18,47.50,48.00,28.00


From the information above, we can infer that **Home Appliances and Electronics products have the highest order value**. Therefore, it is recommended that we push for more conversions in these products since they have high returns. However, it should be noted that this **strategy poses high risk**, especially for the **Home Appliances category** with its **high cancellation rate (40%)**. The cost of conversion might not be recovered if the cancellation trend persists. This means it is **safer to convert consumers in purchasing electronics products**, considering their **lower cancellation rate (27.97%)** and relatively high order value **($1,215.29)**.

In conclusion, different strategies could be chosen depending on how high the platform's risk tolerance is.

In [29]:
#Product Metric

Product = data.groupby(['Category','Product']).agg(
    Total_Expected_Revenue = ('Expected Revenue','sum'),
    Total_Earned_Revenue = ('Earned Revenue', 'sum'),
    Total_Deferred_Revenue = ('Deferred Revenue','sum'),
    Cancelled_Revenue = ('Cancelled Revenue', 'sum'),
    Expected_Item_Purchased = ('Expected Quantity','sum'),
    Order = ('Order ID','size'),
    Completion_Rate = ('Status', lambda x : ((x == 'Completed').sum()/len(x)*100).round(2)),
    Cancellation_Rate = ('Status', lambda x : ((x == 'Cancelled').sum()/len(x)*100).round(2)),
    Pending_Rate = ('Status', lambda x : ((x == 'Pending').sum()/len(x)*100).round(2))).reset_index()

Product['Price Per Item'] = Product['Total_Expected_Revenue']/Product['Expected_Item_Purchased']
Product = Product[['Category',
 'Product',
 'Total_Expected_Revenue',
 'Total_Earned_Revenue',
 'Total_Deferred_Revenue',
 'Cancelled_Revenue',
 'Expected_Item_Purchased',
 'Price Per Item', 'Order',
 'Completion_Rate',
 'Cancellation_Rate',
 'Pending_Rate']]

(
    Product
    .sort_values('Total_Expected_Revenue', ascending = False).reset_index(drop = True)
    [['Category','Product','Total_Expected_Revenue','Expected_Item_Purchased','Price Per Item', 'Pending_Rate', 'Cancellation_Rate']]
)

,Category,Product,Total_Expected_Revenue,Expected_Item_Purchased,Price Per Item,Pending_Rate,Cancellation_Rate
0,Home Appliances,Refrigerator,54000,45,1200.0,29.17,33.33
1,Electronics,Laptop,48800,61,800.0,41.67,20.83
2,Electronics,Smartphone,39000,78,500.0,34.29,25.71
3,Home Appliances,Washing Machine,15000,25,600.0,25.00,50.00
4,Electronics,Smartwatch,11400,76,150.0,32.35,26.47
5,Electronics,Headphones,4100,41,100.0,32.00,40.00
6,Footwear,Running Shoes,3240,54,60.0,33.33,29.63
7,Clothing,Jeans,1520,38,40.0,25.00,45.00
8,Clothing,T-Shirt,900,45,20.0,35.00,20.00
9,Books,Book,855,57,15.0,48.00,28.00


As we can see, **both Home Appliances and Electronics products are expected to bring in the most revenue.** The more lucrative products are Refrigerator (\\$1,200/item), Laptop (\\$800/item), and Smartphone ($500/item). Considering their relatively high pending rates, orders for these products could be prioritized for completion.

**Previous analysis shows** that the **Home Appliances category has a high cancellation rate**. From the table above, we can see that the more profitable product (Refrigerator) has a relatively decent cancellation rate (33.33%). **What causes the high cancellation rate in the same category is actually the less profitable product (Washing Machine) with a 50% cancellation rate**. Hence, our conversion strategy could be revised from converting all Home Appliances buyers to specifically targeting Refrigerator buyers.

-------------------------------------------------------------------------------------------------------------------------------

**3. Time-Based Analysis**

Here is the thought process for the dataframe created:

* **Monthly Time Analysis** - To understand the platform's monthly performance by order and revenue

* **Product By Month** - To identify how products affect the platform's monthly performance

By analyzing the platform's monthly performance, we can identify period-based buying patterns and the sensitivity of each product's impact on platform revenue. With this information, we can adjust our strategy according to product sensitivity and the current period.

In [33]:
#Time Analysis
Monthly_Time_Analysis = data.groupby(['Month']).agg(
    Order = ('Order ID', 'size'),
    Expected_Revenue = ('Expected Revenue','sum'),
    Earned_Revenue = ('Earned Revenue','sum'),
    Pending_Order = ('Pending Order', 'sum'),
    Cancelled_Order = ('Cancelled Order', 'sum')).reset_index()

Monthly_Time_Analysis

,Month,Order,Expected_Revenue,Earned_Revenue,Pending_Order,Cancelled_Order
0,2,113,91425,40865,39,38
1,3,131,84770,45145,45,37
2,4,6,2620,2520,1,2


From the data above, we can see that **despite the rise in orders**, the **expected revenue still went down compared to the previous month**. This means there was a decrease in orders for high-value products and an increase in orders for low-value products. To identify which products saw a decrease or increase in orders, we can refer to the table below.

In [35]:
Product_By_Month = (
    data
    .groupby(['Month','Product'])['Expected Quantity'].sum().reset_index()
    .pivot(columns = 'Month', index = 'Product', values = 'Expected Quantity').fillna(0)
    .rename_axis(None, axis = 1).reset_index()
    .merge(Product[['Product','Price Per Item']], how = 'left', on = 'Product').iloc[:,[0,4,1,2,3]]
)

Product_By_Month

,Product,Price Per Item,2,3,4
0,Book,15.0,21.0,36.0,0.0
1,Headphones,100.0,11.0,30.0,0.0
2,Jeans,40.0,16.0,22.0,0.0
3,Laptop,800.0,29.0,29.0,3.0
4,Refrigerator,1200.0,26.0,19.0,0.0
5,Running Shoes,60.0,17.0,37.0,0.0
6,Smartphone,500.0,44.0,34.0,0.0
7,Smartwatch,150.0,33.0,43.0,0.0
8,T-Shirt,20.0,20.0,14.0,11.0
9,Washing Machine,600.0,11.0,14.0,0.0


As we can see, high value products (Refrigerator, Smartphone) saw a decrease in order, dropping the platform overall expected revenue. Therefore, to maintain platform's profit, it is neccesary to **mantain the order quantity for the high value products such as Refrigerator and Smartphone** with means that have been stated previously (discount, promo, etc).

--------------------------------------------------------------------------------------------------------------------------------

**4. Payment Method Analysis**

Here is the thought process for the dataframe created:

* **Payment Method Analysis** - To understand the correlation between payment method and purchasing behavior on the platform.

Analyzing payment methods will help us understand the correlation between payment method and consumer behavior. With this information, we can **decide which payment methods could be incentivized or restricted.**

In [39]:
Payment_Method_Analysis =  data.groupby('Payment Method').agg(
    Order = ('Order ID', 'size'),
    Completion_Rate = ('Status', lambda x : ((x == 'Completed').sum()/len(x)*100).round(2)),
    Cancellation_Rate = ('Status', lambda x : ((x == 'Cancelled').sum()/len(x)*100).round(2)),
    Pending_Rate = ('Status', lambda x : ((x == 'Pending').sum()/len(x)*100).round(2))).reset_index()

Payment_Method_Analysis.sort_values('Completion_Rate', ascending = False)

,Payment Method,Order,Completion_Rate,Cancellation_Rate,Pending_Rate
0,Amazon Pay,41,51.22,17.07,31.71
4,PayPal,60,50.00,26.67,23.33
1,Credit Card,54,31.48,29.63,38.89
2,Debit Card,53,26.42,37.74,35.85
3,Gift Card,42,14.29,42.86,42.86


From the data above, we can see orders paid using **Amazon Pay** have the **highest completion rate (51.22%)**, followed by PayPal (50%). On the other hand, **Gift Card** has the **lowest completion rate (14.29%)** and the **highest cancellation rate (42.86%)**.

With this information, we could **push for more payments using Amazon Pay or PayPal**. Incentives could include **free admin fees** when **topping up** Amazon Pay or **cashback** for purchases made through Amazon Pay or PayPal. For gift cards, we could restrict gift card payments to only specific categories or products.

Through these measures, we hope to **decrease the cancellation rate**.

-------------------------------------------------------------------------------------------------------------------------------------

**Key Takeaways**

From the analysis above, we can conclude several key insights:

* The **platform's orders** are facing a **high cancellation rate (30.8% of total transactions)**, resulting in a **loss of 42.3% of potential revenue**. Lowering the order cancellation rate should be a top priority for this platform.

* The platform should **prioritize converting customers with a high volume of pending orders** (e.g., Olivia Wilson) and ensure they follow through with their transactions to maximize earnings. **Customers** with **high cancellation rates** (e.g., David Lee, Jane Smith) can **serve as reference points** to understand the **root causes of cancellations**.

* **Electronics and Home Appliances** (specifically Refrigerators) are **categories that generate the highest order value and purchases**. Therefore, the platform should focus on converting customers to purchase Electronics and Home Appliances (particularly Refrigerators).

* The platform should **incentivize** customers to pay **using Amazon Pay or PayPal**, while also **restricting gift card** availability.

All recommendations above are important steps toward maximizing the platform's revenue and should be considered priorities moving forward.